In [16]:
from parsers.lr.lr1 import LR1Parser
from parsers.lr.helpers import get_conflicts
from parsers.lr.lr_parsing_engine import LREngine

from parsers.grammars.appel_grammar_parser import appel_to_ruleset

In [17]:
grammar = [
    "P -> L",
    "S -> id := id",
    "S -> if id then S",
    "S -> if id then S else S",
    "L -> S",
    "L -> L ; S",
]

In [18]:
p = LR1Parser(appel_to_ruleset(grammar)).to_lalr()

2026-05-26 01:39:02.142 | DEBUG    | parsers.base_parser:__init__:34 - RuleSet(start_rule_idx=0, rules={0: RuleType(lhs='P', rhs=('L', '$end')), 1: RuleType(lhs='S', rhs=('id', ':=', 'id')), 2: RuleType(lhs='S', rhs=('if', 'id', 'then', 'S')), 3: RuleType(lhs='S', rhs=('if', 'id', 'then', 'S', 'else', 'S')), 4: RuleType(lhs='L', rhs=('S',)), 5: RuleType(lhs='L', rhs=('L', ';', 'S'))}, eol='$end')


In [22]:
p.print_rules_and_states()

---=== Rules ===---
0000 P -> L $end
0001 S -> id := id
0002 S -> if id then S
0003 S -> if id then S else S
0004 L -> S
0005 L -> L ; S

---=== States ===---
state 0
    S -> . if id then S ($end ;)
    L -> L ; . S ($end ;)
    S -> . if id then S else S ($end ;)
    S -> . id := id ($end ;)

state 1
    S -> if id . then S ($end ; else)
    S -> if id . then S else S ($end ; else)

state 2
    L -> L . ; S ($end ;)
    P -> L . $end

state 3
    S -> id := . id ($end ; else)

state 4
    S -> if id then . S else S ($end ; else)
    S -> . if id then S ($end ; else)
    S -> if id then . S ($end ; else)
    S -> . if id then S else S ($end ; else)
    S -> . id := id ($end ; else)

state 6
    S -> . if id then S ($end ; else)
    S -> if id then S else . S ($end ; else)
    S -> . if id then S else S ($end ; else)
    S -> . id := id ($end ; else)

state 7
    S -> if id then S else S . ($end ; else)

state 8
    S -> id := id . ($end ; else)

state 9
    S -> id . := id ($end ; els

In [20]:
print(p.to_tabulate())

┌────┬────────┬──────┬────────┬─────┬────────┬──────┬──────┬─────┬─────┬─────┐
│    │ then   │ :=   │ else   │ ;   │ $end   │ if   │ id   │ S   │ L   │ P   │
├────┼────────┼──────┼────────┼─────┼────────┼──────┼──────┼─────┼─────┼─────┤
│  0 │        │      │        │     │        │ s15  │ s9   │ g10 │     │     │
├────┼────────┼──────┼────────┼─────┼────────┼──────┼──────┼─────┼─────┼─────┤
│  1 │ s4     │      │        │     │        │      │      │     │     │     │
├────┼────────┼──────┼────────┼─────┼────────┼──────┼──────┼─────┼─────┼─────┤
│  2 │        │      │        │ s0  │ a      │      │      │     │     │     │
├────┼────────┼──────┼────────┼─────┼────────┼──────┼──────┼─────┼─────┼─────┤
│  3 │        │      │        │     │        │      │ s8   │     │     │     │
├────┼────────┼──────┼────────┼─────┼────────┼──────┼──────┼─────┼─────┼─────┤
│  4 │        │      │        │     │        │ s15  │ s9   │ g12 │     │     │
├────┼────────┼──────┼────────┼─────┼────────┼──────

In [21]:
len(p.parsing_table)

14

In [ ]:
conflicts = get_conflicts(p.parsing_table)
state, sym, _ = conflicts[0]

cell = list(p.parsing_table[state][sym])
p.parsing_table[state][sym] = set(cell[1:])

In [ ]:
print(p.to_tabulate())

In [ ]:
p.print_rules_and_states()

In [ ]:
e = LREngine(p)

In [ ]:
code_sample = """
if id then     
    if id then
        id := id
    else
        id := id ;
        id := id $end
""".split()

In [ ]:
def print_iter(stack, states, symbol, action):
    for el in stack:
        print_el = next(iter(el.keys())) if isinstance(el, dict) else el
        print(print_el, end=", ")
    print(f" {symbol=}, {states[-1]=}, {action=}")

In [ ]:
import json

In [ ]:
out = e.parse(code_sample, iteration_callback=print_iter)

In [ ]:
from parsers.lr.helpers import pretify_stack

In [ ]:
print(pretify_stack(out))

In [ ]:
print(json.dumps(out, indent=2))